# 🛫 Customer Churn Prediction using Random Forest
## End-to-End Machine Learning and Deployment Use Case

**Course:** B.Tech – Gen AI (2nd Semester)  
**Dataset:** Customertravel.csv  
**Model File:** model_ibm.pkl

---

## 1. Introduction

### What is Customer Churn?
**Customer churn** refers to the phenomenon where customers stop doing business with a company. In the travel industry, churn happens when customers stop booking services, switch to competitors, or simply disengage from the platform.

### Why is Churn Prediction Important?
- **Keeping customers is cheaper** – Getting a new customer costs 5–7 times more than retaining an existing one
- **Take action early** – Predict churn before it happens and offer timely incentives
- **More revenue** – Retained customers spend more over time

### Why Random Forest?
- ✅ Handles both numeric and categorical data
- ✅ Avoids overfitting via ensemble of trees
- ✅ Provides feature importance scores
- ✅ Works well with imbalanced datasets
- ✅ No need for feature scaling

---

## 2. Importing Required Libraries

In [ ]:
# Install scikit-plot (optional — only needed if using skplt)
!pip install scikit-plot --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_curve, auc
)

print("✅ All libraries imported successfully!")

## 3. Data Loading and Exploration

> **Note:** The original `Customertravel.csv` file is actually a **PDF document** with the data embedded as tables. Run the cell below to extract it properly, or simply place a correctly formatted `Customertravel_clean.csv` in the same folder.

In [ ]:
# ── Option A: Load from pre-cleaned CSV (recommended) ──────────────────────
# Place Customertravel_clean.csv in the same folder and run this cell.
import os

if os.path.exists('Customertravel_clean.csv'):
    df = pd.read_csv('Customertravel_clean.csv')
    print("✅ Loaded from Customertravel_clean.csv")

elif os.path.exists('Customertravel.csv'):
    # ── Option B: Auto-extract from PDF-disguised CSV ─────────────────────
    try:
        import pdfplumber
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', 'pdfplumber', '-q'])
        import pdfplumber

    feature_rows, target_rows = [], []
    with pdfplumber.open('Customertravel.csv') as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()
            if not tables:
                continue
            t = tables[0]
            first = t[0][0] if t[0] else ''
            if len(t[0]) == 6:           # Feature columns
                for row in t:
                    if row[0] == 'Age':
                        continue
                    feature_rows.append(row)
            elif len(t[0]) == 1:         # Target column
                for row in t:
                    if row[0] == 'Target':
                        continue
                    target_rows.append(row)

    cols = ['Age','FrequentFlyer','AnnualIncomeClass',
            'ServicesOpted','AccountSyncedToSocialMedia','BookedHotelOrNot']
    df = pd.DataFrame(feature_rows, columns=cols)
    df['Target'] = [r[0] for r in target_rows]
    df.to_csv('Customertravel_clean.csv', index=False)
    print("✅ Extracted from PDF and saved as Customertravel_clean.csv")

else:
    raise FileNotFoundError("⚠️  Place Customertravel.csv in this folder and rerun.")

print(f"📊 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)

In [ ]:
# Dataset dimensions and column names
print(f'📐 Dimensions: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\n📋 Columns:')
for col in df.columns:
    print(f'   - {col}')

In [ ]:
# Summary statistics
print('📈 Summary Statistics:')
df.describe(include='all')

In [ ]:
# Data types and non-null counts
print('🔍 Data Types & Non-Null Counts:')
df.info()

In [ ]:
# Missing values check
missing = df.isnull().sum()
if missing.sum() == 0:
    print('✅ No missing values found in the dataset!')
else:
    print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing/len(df)*100}))

In [ ]:
# Unique values per column
print('🔢 Unique Values per Column:')
for col in df.columns:
    print(f'   {col}: {df[col].nunique()} unique → {list(df[col].unique()[:5])}')

## 4. Data Cleaning and Preprocessing

In [ ]:
# Step 1: Make a clean copy & strip column name whitespace
df_clean = df.copy()
df_clean.columns = df_clean.columns.str.strip()
print('✅ Column names cleaned:', list(df_clean.columns))

In [ ]:
# Step 2: Handle 'No Record' in FrequentFlyer → replace with 'No'
print('FrequentFlyer counts before:', df_clean['FrequentFlyer'].value_counts().to_dict())
df_clean['FrequentFlyer'] = df_clean['FrequentFlyer'].replace('No Record', 'No')
print('FrequentFlyer counts after :', df_clean['FrequentFlyer'].value_counts().to_dict())

In [ ]:
# Step 3: Label-encode all categorical columns
# Encoding map used:
#   FrequentFlyer          : No=0, Yes=1
#   AnnualIncomeClass      : High Income=0, Low Income=1, Middle Income=2
#   AccountSyncedToSocialMedia : No=0, Yes=1
#   BookedHotelOrNot       : No=0, Yes=1

label_encoders = {}
categorical_cols = [
    'FrequentFlyer', 'AnnualIncomeClass',
    'AccountSyncedToSocialMedia', 'BookedHotelOrNot'
]

for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    print(f'✅ Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

In [ ]:
# Step 4: Separate features (X) and target (y)
X = df_clean.drop(columns=['Target'])
y = df_clean['Target'].astype(int)

print(f'✅ Features (X) shape : {X.shape}')
print(f'✅ Target  (y) shape  : {y.shape}')
print(f'\n📊 Target Distribution:')
print(y.value_counts().rename({0:'Not Churned', 1:'Churned'}))
print(f'\nChurn Rate: {y.mean()*100:.2f}%')

In [ ]:
# Step 5: Train-Test Split (80 : 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'✅ Training set : {X_train.shape[0]} samples')
print(f'✅ Testing set  : {X_test.shape[0]} samples')

## 5. Model Development: Random Forest Classifier

> We load the **pre-trained `model_ibm.pkl`** file (Random Forest, trained on the same 6 features). This avoids retraining and matches the Streamlit deployment exactly.

In [ ]:
# Load the pre-trained model (model_ibm.pkl)
with open('model_ibm.pkl', 'rb') as f:
    rf_model = pickle.load(f)

print('✅ model_ibm.pkl loaded successfully!')
print(f'   Algorithm      : {type(rf_model).__name__}')
print(f'   n_estimators   : {rf_model.n_estimators}')
print(f'   max_depth      : {rf_model.max_depth}')
print(f'   Features used  : {list(rf_model.feature_names_in_)}')

# Generate predictions on test set
y_pred       = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)

print(f'\n✅ Predictions generated for {len(y_test)} test samples.')

## 6. Model Evaluation

In [ ]:
# 6.1  Accuracy
train_acc = accuracy_score(y_train, rf_model.predict(X_train))
test_acc  = accuracy_score(y_test,  y_pred)

print('=' * 45)
print('         MODEL ACCURACY SCORES')
print('=' * 45)
print(f'  Training Accuracy : {train_acc * 100:.2f}%')
print(f'  Test Accuracy     : {test_acc  * 100:.2f}%')
print('=' * 45)

In [ ]:
# 6.2  Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Churned', 'Churned'],
            yticklabels=['Not Churned', 'Churned'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix – Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\n  True Negatives  (Correctly Not Churned) : {tn}')
print(f'  False Positives (Wrongly flagged Churned): {fp}')
print(f'  False Negatives (Missed Churners)        : {fn}')
print(f'  True Positives  (Correctly Churned)      : {tp}')

In [ ]:
# 6.3  Classification Report
print('📋 CLASSIFICATION REPORT:')
print('=' * 60)
print(classification_report(y_test, y_pred,
      target_names=['Not Churned', 'Churned']))
print('=' * 60)

In [ ]:
# 6.4  ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#1565C0', lw=2.5,
         label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--',
         label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.15, color='#1565C0')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve – Random Forest', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()
print(f'✅ AUC Score: {roc_auc:.4f}')

In [ ]:
# 6.5  Feature Importance
feature_importances = pd.Series(
    rf_model.feature_importances_,
    index=rf_model.feature_names_in_
).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(
    x=feature_importances.values,
    y=feature_importances.index,
    palette='Blues_r',
    hue=feature_importances.index,
    legend=False
)
plt.title('Feature Importance – Random Forest', fontweight='bold', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print('\n📊 Feature Importance Scores:')
for feat, val in feature_importances.items():
    print(f'   {feat:<35} {val*100:.2f}%')

## 7. Visualizations

In [ ]:
# 7.1  Churn Distribution
plt.figure(figsize=(6, 4))
churn_counts = y.value_counts()
colors = ['#42A5F5', '#EF5350']
bars = plt.bar(['Not Churned', 'Churned'], churn_counts.values, color=colors,
               edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, churn_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 8, str(val),
             ha='center', fontweight='bold', fontsize=11)
plt.title('Churn Distribution in Dataset', fontweight='bold', fontsize=13)
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150)
plt.show()
print(f'Not Churned: {churn_counts[0]} | Churned: {churn_counts[1]} | Rate: {y.mean()*100:.1f}%')

In [ ]:
# 7.2  Age Distribution by Churn
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Age KDE by Churn
df_plot = df.copy()
df_plot['Churn Label'] = df_plot['Target'].map({0: 'Not Churned', 1: 'Churned'})
for label, color in zip(['Not Churned', 'Churned'], ['#42A5F5', '#EF5350']):
    subset = df_plot[df_plot['Churn Label'] == label]['Age']
    axes[0].hist(subset, alpha=0.6, label=label, color=color, bins=10, edgecolor='white')
axes[0].set_title('Age Distribution by Churn', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

# Services Opted by Churn
churn_services = df_plot.groupby(['ServicesOpted','Churn Label']).size().unstack(fill_value=0)
churn_services.plot(kind='bar', ax=axes[1], color=['#42A5F5','#EF5350'],
                   edgecolor='white', linewidth=0.8)
axes[1].set_title('Services Opted by Churn', fontweight='bold')
axes[1].set_xlabel('Services Opted')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('feature_analysis.png', dpi=150)
plt.show()

## 8. Model Verification with model_ibm.pkl

In [ ]:
# Verify the loaded model predicts correctly on new data
# Sample: Age=30, FrequentFlyer=Yes(1), AnnualIncome=High(0),
#         ServicesOpted=5, AccountSynced=No(0), BookedHotel=Yes(1)
sample = np.array([[30, 1, 0, 5, 0, 1]])

pred   = rf_model.predict(sample)[0]
proba  = rf_model.predict_proba(sample)[0]

print('🔮 Sample Prediction:')
print(f'   Prediction   : {"CHURN" if pred == 1 else "NOT CHURN"}')
print(f'   Churn Prob   : {proba[1]*100:.1f}%')
print(f'   No-Churn Prob: {proba[0]*100:.1f}%')

## 9. Streamlit App Code (app.py)

The `app.py` file below is the deployment-ready Streamlit app. Place it alongside `model_ibm.pkl` and `requirements.txt` in your GitHub repository.

In [ ]:
app_code = '''
import streamlit as st
import pickle
import numpy as np

st.set_page_config(
    page_title="Customer Churn Predictor",
    page_icon="✈️",
    layout="centered"
)

@st.cache_resource
def load_model():
    with open("model_ibm.pkl", "rb") as f:
        return pickle.load(f)

model = load_model()

st.title("✈️ Customer Churn Predictor")
st.markdown("**Predict whether a travel customer will churn using Random Forest (model_ibm.pkl).**")
st.divider()

st.subheader("🧾 Enter Customer Details")
col1, col2 = st.columns(2)

with col1:
    age             = st.slider("Age", 18, 65, 30)
    frequent_flyer  = st.selectbox("Frequent Flyer", ["No", "Yes"])
    annual_income   = st.selectbox("Annual Income Class",
                                   ["High Income", "Low Income", "Middle Income"])
with col2:
    services_opted  = st.slider("Services Opted", 1, 6, 3)
    account_synced  = st.selectbox("Account Synced to Social Media", ["No", "Yes"])
    booked_hotel    = st.selectbox("Booked Hotel", ["No", "Yes"])

# Encode — must match LabelEncoder order used during training
ff_enc     = 1 if frequent_flyer == "Yes" else 0
income_map = {"High Income": 0, "Low Income": 1, "Middle Income": 2}
ai_enc     = income_map[annual_income]
as_enc     = 1 if account_synced == "Yes" else 0
bh_enc     = 1 if booked_hotel  == "Yes" else 0

input_data = np.array([[age, ff_enc, ai_enc, services_opted, as_enc, bh_enc]])

st.divider()
if st.button("🔮 Predict Churn", type="primary", use_container_width=True):
    prediction = model.predict(input_data)[0]
    proba      = model.predict_proba(input_data)[0]

    st.subheader("📊 Prediction Result")
    if prediction == 1:
        st.error(f"🚨 **Customer is likely to CHURN** (Confidence: {proba[1]*100:.1f}%)")
        st.markdown("💡 **Recommendation:** Offer personalized discounts or loyalty rewards.")
    else:
        st.success(f"✅ **Customer is NOT likely to churn** (Confidence: {proba[0]*100:.1f}%)")
        st.markdown("💡 **Recommendation:** Continue current engagement strategy.")

    st.metric("Churn Probability", f"{proba[1]*100:.1f}%")
'''

with open('app.py', 'w') as f:
    f.write(app_code.strip())
print('✅ app.py saved!')

## 10. Conclusion

### Model Performance Summary

In [ ]:
print('=' * 60)
print('                  PROJECT SUMMARY')
print('=' * 60)
print(f'  Dataset Size        : {df.shape[0]} samples, {df.shape[1]} features')
print(f'  Train / Test Split  : 80% / 20%')
print(f'  Model               : Random Forest (model_ibm.pkl)')
print(f'  Training Accuracy   : {train_acc * 100:.2f}%')
print(f'  Test Accuracy       : {test_acc  * 100:.2f}%')
print(f'  AUC Score           : {roc_auc:.4f}')
print('=' * 60)

print(f'\n🏆 Top Features Influencing Churn:')
for i, (feat, val) in enumerate(feature_importances.head(3).items(), 1):
    print(f'   {i}. {feat}: {val*100:.2f}%')

print('\n💡 Key Insights:')
print('   • ServicesOpted and Age are the strongest churn predictors.')
print('   • Frequent Flyers tend to have lower churn rates.')
print('   • High-income customers churn less than low-income customers.')

print('\n🚀 Future Improvements:')
print('   • Hyperparameter tuning with GridSearchCV.')
print('   • Try XGBoost or LightGBM for comparison.')
print('   • Apply SMOTE for better handling of class imbalance.')
print('   • Collect more features (e.g., customer tenure, feedback score).')
print('=' * 60)